# E044 — Multi-prompt Best Pipeline + Audit complet du papier DiffQRCoder

Ce notebook est le livrable d'analyse principal. Il compare **papier v3**, **code public officiel** et **pipeline Prooftag E044**, puis analyse 7 familles de prompts, deux gamma SR-MPGD et tous les checkpoints i0→i8.

**Important :** E044 n'est pas présenté comme une reproduction exacte du papier. Le notebook sépare ce qui est explicitement décrit, ce que fait le code public et les choix Prooftag.


In [ ]:
from pathlib import Path
import json, math, pandas as pd, numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image
R=Path('/data/e044-multi-prompt-best-pipeline-v1')
ROOT=Path('/workspace')
verdict=json.loads((R/'verdict.json').read_text(encoding='utf-8'))
rows=json.loads((R/'comparison-all.json').read_text(encoding='utf-8'))
summary=json.loads((R/'prompt-summary.json').read_text(encoding='utf-8'))
audit=json.loads((ROOT/'docs/e044-paper-audit.json').read_text(encoding='utf-8'))
catalog=json.loads((ROOT/'docs/e044-prompt-catalog.json').read_text(encoding='utf-8'))
df=pd.DataFrame(rows)
display(verdict)


## 1. Ce que nous utilisons exactement

### Pipeline E044

1. **Stage 1** : Cetus-Mix Whalefall + QR Monster v2, 40 pas, CFG 7.5, ControlNet 1.35.
2. **Stage 2 SRPG** : `public_random`, cible `binary_exact`, 40 pas, ControlNet 1.05, poids SRL/Perceptuel **50/20**.
3. **SR-MPGD** : `scanaware_v2`, rayon latent RMS 0.20, checkpoints i0..i8, **gamma 500 et 1000**, LPIPS VGG λ=0.01, trust-region + backtracking.
4. **Raster** : 736×736, padding exact 78, cœur 580×580, 29×29 modules de 20 px.
5. **Scanner autoritaire** : `antfu/qr-verify`, 37 presets, répétitions conservatrices et payload exact.
6. **Qualité** : LPIPS, CLIPScore, CLIP-Aesthetic, HPSv2, MAE, clipping, saturation.

Les autres décodeurs (OpenCV, ZBar, ZXing-C++, OpenCV WeChat) sont diagnostiques seulement.


In [ ]:
display(pd.DataFrame([audit['e044_setup']]).T.rename(columns={0:'E044'}))


## 2. Papier officiel vs code public vs E044

Le papier v3 rapporte notamment : 100 prompts GPT-4, `easynegative`, QR v3/M/mask4, module 20 px, **padding 80**, deux stages de 40 pas, Stage 2 conditionné par **QArt**, λ1=500, meilleure configuration λ2=3, puis SR-MPGD γ=1000 et λLPIPS=0.01.

Le code public distribué n'est pas identique sur plusieurs détails : padding par défaut 78, runner perceptuel=2, signature pipeline perceptuelle=10, SRL avec seuils 0.45/0.65, sigma gaussien 1.5, et SR-MPGD implémenté par SGD direct sur le latent.

E044 conserve volontairement notre meilleur protocole expérimental reproductible plutôt que de prétendre reconstruire le QArt privé/exact des auteurs.


In [ ]:
paper=pd.Series(audit['paper_reported_setup'],name='Papier').to_frame()
public=pd.Series(audit['official_public_code_observations'],name='Code public').to_frame()
e044=pd.Series(audit['e044_setup'],name='E044').to_frame()
display(paper); display(public); display(e044)


## 3. Zones floues / insuffisamment reproductibles dans le papier

Ici « flou » signifie : le PDF seul ne permet pas une reconstruction byte-à-byte ou laisse une décision technique importante à l'implémenteur.


In [ ]:
ambiguities=pd.DataFrame(audit['ambiguities_or_reproducibility_gaps'])
display(ambiguities.style.set_properties(subset=['detail'], **{'white-space':'normal'}))


## 4. Catalogue des prompts E044


In [ ]:
display(pd.DataFrame(catalog['prompts'])[['id','family','text','reason']].style.set_properties(subset=['text','reason'],**{'white-space':'normal'}))


## 5. Résumé des meilleurs résultats par prompt


In [ ]:
sdf=pd.DataFrame(summary)
display(sdf[['prompt_id','family','best_gamma','best_iteration','best_ssr_exact_presets','best_original_exact','best_lpips','best_clip_aesthetic','best_module_errors']])


## 6. Atlas des gagnants par prompt


In [ ]:
for s in summary:
    display(Markdown(f"### {s['prompt_id']} — {s['family']} — g={int(s['best_gamma'])}, i{s['best_iteration']}, SSR={s['best_ssr_exact_presets']}/37"))
    display(Image(filename=s['best_image_path'], width=520))


## 7. SSR maximum par prompt et gamma


In [ ]:
tmp=df[(df['gamma'].isin([500.0,1000.0])) & (df['visual_guard_pass']==True)].groupby(['prompt_id','gamma'])['qr_verify_exact_presets'].max().unstack(fill_value=0)
tmp.plot(kind='bar',figsize=(13,5)); plt.ylabel('QR-Verify exact presets / 37'); plt.title('Meilleur SSR visuellement sûr par prompt et gamma'); plt.tight_layout(); plt.show()


## 8. Trajectoires SSR i0→i8 pour chaque prompt


In [ ]:
for pid in [p['id'] for p in catalog['prompts']]:
    sub=df[(df.prompt_id==pid)&(df.gamma.isin([500.0,1000.0]))]
    fig,ax=plt.subplots(figsize=(10,4))
    for g in [500.0,1000.0]:
        q=sub[sub.gamma==g].sort_values('iteration'); ax.plot(q.iteration,q.qr_verify_exact_presets,marker='o',label=f'gamma {int(g)}')
    ax.set_title(pid); ax.set_xlabel('iteration'); ax.set_ylabel('exact presets /37'); ax.legend(); ax.grid(alpha=.2); plt.show()


## 9. MER / erreurs modules le long des trajectoires


In [ ]:
for pid in [p['id'] for p in catalog['prompts']]:
    sub=df[(df.prompt_id==pid)&(df.gamma.isin([500.0,1000.0]))]
    fig,ax=plt.subplots(figsize=(10,4))
    for g in [500.0,1000.0]:
        q=sub[sub.gamma==g].sort_values('iteration'); ax.plot(q.iteration,q.full_module_error_count,marker='o',label=f'gamma {int(g)}')
    ax.set_title(pid+' — module errors'); ax.set_xlabel('iteration'); ax.set_ylabel('error modules'); ax.legend(); ax.grid(alpha=.2); plt.show()


## 10. LPIPS et frontière esthétique


In [ ]:
safe=df[df.visual_guard_pass==True]
fig,ax=plt.subplots(figsize=(9,6)); ax.scatter(df.lpips,df.qr_verify_exact_presets,alpha=.35,label='tous'); ax.scatter(safe.lpips,safe.qr_verify_exact_presets,alpha=.65,label='visual-safe'); ax.set_xlabel('LPIPS'); ax.set_ylabel('SSR exact presets /37'); ax.set_title('SSR vs LPIPS'); ax.legend(); ax.grid(alpha=.2); plt.show()


## 11. CLIP-Aesthetic, CLIPScore et HPS


In [ ]:
metrics=['clip_aesthetic','clip_score','hpsv2_1']
for m in metrics:
    if m in df and df[m].notna().any():
        fig,ax=plt.subplots(figsize=(9,5)); ax.scatter(df[m],df.qr_verify_exact_presets,alpha=.45); ax.set_xlabel(m); ax.set_ylabel('SSR /37'); ax.set_title(f'SSR vs {m}'); ax.grid(alpha=.2); plt.show()


## 12. La métrique interne prédit-elle réellement QR-Verify ?


In [ ]:
cols=[c for c in ['qr_verify_exact_presets','full_module_error_count','mean_total_error_count','mean_data_error_count','mean_format_error_count','intra_module_std_p90','lpips','clip_aesthetic','clip_score','hpsv2_1'] if c in df]
corr=df[cols].corr(numeric_only=True)
fig,ax=plt.subplots(figsize=(10,8)); im=ax.imshow(corr.values,vmin=-1,vmax=1,cmap='coolwarm'); ax.set_xticks(range(len(cols)),cols,rotation=70,ha='right'); ax.set_yticks(range(len(cols)),cols); fig.colorbar(im,ax=ax); ax.set_title('Corrélations'); plt.tight_layout(); plt.show(); display(corr)


## 13. Projection et backtracking : gamma 500 vs 1000


In [ ]:
proj=df[df.gamma.isin([500.0,1000.0])].groupby('gamma').agg(checkpoints=('variant','count'),projection_active=('projection_was_active','sum'),accepted_alpha_mean=('accepted_alpha','mean'),rejections_mean=('rejected_trial_count','mean'),ssr_max=('qr_verify_exact_presets','max'))
display(proj)


In [ ]:
fig,ax=plt.subplots(figsize=(7,4)); proj['projection_active'].plot(kind='bar',ax=ax); ax.set_ylabel('checkpoints avec projection active'); ax.set_title('Saturation trust-region par gamma'); plt.tight_layout(); plt.show()


## 14. Pourquoi les checkpoints sortent de la garde esthétique ?


In [ ]:
from collections import Counter
fails=Counter()
for checks in df.loc[~df.visual_guard_pass,'visual_guard_checks']:
    if isinstance(checks,dict):
        for k,v in checks.items():
            if not v: fails[k]+=1
fd=pd.DataFrame(fails.items(),columns=['check','fail_count']).sort_values('fail_count',ascending=False)
display(fd)
if len(fd): fd.set_index('check')['fail_count'].plot(kind='bar',figsize=(11,4)); plt.ylabel('échecs'); plt.title('Critères de garde qui cassent le plus'); plt.tight_layout(); plt.show()


## 15. Contact sheet global


In [ ]:
display(Image(filename=str(R/'pipeline/best-by-prompt-contact-sheet.png')))


## 16. Table complète de tous les états

Cette table contient Stage 2 + tous les checkpoints SR-MPGD. Elle permet d'auditer le classement sans masquer les échecs.


In [ ]:
show=[c for c in ['prompt_id','variant','gamma','iteration','qr_verify_exact_presets','original_exact','visual_guard_pass','full_module_error_count','mean_data_error_count','mean_format_error_count','intra_module_std_p90','lpips','clip_score','clip_aesthetic','hpsv2_1','latent_delta_rms','accepted_alpha','projection_was_active','rejected_trial_count'] if c in df]
display(df[show].sort_values(['prompt_id','gamma','iteration'],na_position='first'))


## 17. Conclusion expérimentale

Le verdict E044 ne sera interprété qu'après lecture de la **sensibilité au prompt**. Un excellent résultat isolé ne suffit pas : E044 reste un screen à seed partagé. La prochaine étape ne sera une généralisation multi-seeds que si plusieurs familles produisent un gain QR-Verify sous garde esthétique.


In [ ]:
display(verdict)
